In [ ]:
from anthropic import Anthropic
from anthropic.lib import files_from_dir
import os
from pathlib import Path
client = Anthropic()
github_token = os.environ.get("GITHUB_TOKEN")

In [6]:
import anthropic
print(anthropic.__version__)          # should print 0.116.0
print(hasattr(client.beta, "deployments"))  # should print True


0.116.0
True


In [ ]:
RUBRIC = """# DCF Model Rubric

## Revenue Projections
- Uses historical revenue data from the last 5 fiscal years
- Projects revenue for at least 5 years forward
- Growth rate assumptions are explicitly stated and reasonable

## Cost Structure
- COGS and operating expenses are modeled separately
- Margins are consistent with historical trends or deviations are justified

## Discount Rate
- WACC is calculated with stated assumptions for cost of equity and cost of debt
- Beta, risk-free rate, and equity risk premium are sourced or justified

## Terminal Value
- Uses either perpetuity growth or exit multiple method (stated which)
- Terminal growth rate does not exceed long-term GDP growth

## Output Quality
- All figures are in a single .xlsx file with clearly labeled sheets
- Key assumptions are on a separate "Assumptions" sheet
- Sensitivity analysis on WACC and terminal growth rate is included
"""
Path("/tmp/rubric.md").write_text(RUBRIC)

rubric = client.beta.files.upload(file=Path("/tmp/rubric.md"))
print(f"Uploaded rubric: {rubric.id}")

In [ ]:
uploaded = client.beta.files.upload(
    file=("earth.pdf", open("/Users/chenwei/Documents/MyPC/cw_explore/earth.pdf", "rb"), "application/pdf"),
)
file_id = uploaded.id
print(file_id)

file = client.beta.files.retrieve_metadata(file_id)

client.beta.files.delete(file_id)


In [ ]:
skill = client.beta.skills.create(
    files=files_from_dir(".claude/skills/deep-research"),
)
skill.id

In [ ]:
store = client.beta.memory_stores.create(
    name="User Preferences",
    description="Per-user preferences and project context.",
)
print(store.id)  # memstore_01Hx...

In [ ]:
'''
# List memories in the store
page = client.beta.memory_stores.memories.list(
    store.id,
    path_prefix="/",
)
for item in page.data:
    print(item.type, item.path)

# Read a memory from the store
retrieved = client.beta.memory_stores.memories.retrieve(
    mem.id,
    memory_store_id=store.id,
)
print(retrieved.content)

# Create a memory in the store
mem = client.beta.memory_stores.memories.create(
    store.id,
    path="/preferences/formatting.md",
    content="Always use tabs, not spaces.",
)

# Update a memory in the store
client.beta.memory_stores.memories.update(
    mem.id,
    memory_store_id=store.id,
    path="/archive/2026_q1_formatting.md",
)

# Delete a memory from the store
client.beta.memory_stores.memories.delete(
    mem.id,
    memory_store_id=store.id,
)

# List stores
for s in client.beta.memory_stores.list(include_archived=True):
    print(s.id, s.name, s.archived_at)

# Archive a store
client.beta.memory_stores.archive(store.id)

# Delete a store
client.beta.memory_stores.delete(
    memory_store_id="memory_store_id",
)
'''

In [ ]:
agent = client.beta.agents.create(
    name="Financial Analyst",
    model="claude-opus-4-8",
    system="You are a helpful financial analyst.",
    tools=[
        {"type": "agent_toolset_20260401"},
    ],
    skills=[
        {
            "type": "anthropic",
            "skill_id": "xlsx",
        },
        {
            "type": "custom",
            "skill_id": skill.id,
            "version": "latest",
        },
    ],
)

print(f"Agent ID: {agent.id}, version: {agent.version}")

In [ ]:
# github mcp server and mcp toolset example
'''
agent = client.beta.agents.create(
    name="Code Reviewer",
    model="claude-opus-4-8",
    system="You are a code review assistant with access to GitHub.",
    mcp_servers=[
        {
            "type": "url",
            "name": "github",
            "url": "https://api.githubcopilot.com/mcp/",
        },
    ],
    tools=[
        {"type": "agent_toolset_20260401"},
        {
            "type": "mcp_toolset",
            "mcp_server_name": "github",
        },
    ],
)

session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    resources=[
        {
            "type": "github_repository",
            "url": "https://github.com/strench0923/cw_explore",
            "mount_path": "/workspace/cw_explore",
            "authorization_token": github_token,
        },
    ],
)

# List resources on the session
listed = client.beta.sessions.resources.list(session.id)
repo_resource_id = listed.data[0].id
print(repo_resource_id)  # "sesrsc_01ABC..."

# Preview snapshots, keyed by event id. accumulate_managed_agents_event folds each
# event_start / event_delta into an agent.message snapshot; the buffered
# agent.message replaces it.
previews: dict[str, BetaManagedAgentsAgentMessageEvent] = {}

# Opt in to agent.message previews on this connection
with client.beta.sessions.events.stream(
    session.id, event_deltas=["agent.message"]
) as stream:
    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [{"type": "text", "text": "Describe the repo in one sentence."}],
            },
        ],
    )

    for event in stream:
        match event.type:
            case "event_start":
                snapshot = accumulate_managed_agents_event(None, event)
                if snapshot is not None:
                    previews[event.event.id] = snapshot
                print(f"event_start             {event.event.type} {event.event.id}")
            case "event_delta":
                preview = accumulate_managed_agents_event(previews.get(event.event_id), event)
                if preview is not None:
                    previews[event.event_id] = preview
                    text = "".join(block.text for block in preview.content)
                    print(f"event_delta             preview: {text!r}")
            case "agent.message":
                # The buffered event is the record: it replaces and closes the preview
                preview = accumulate_managed_agents_event(previews.pop(event.id, None), event)
                text = "".join(block.text for block in preview.content)
                print(f"agent.message           {event.id} {text!r}")
            case "span.model_request_end":
                # No more deltas are coming. Close any preview whose
                # buffered event never arrived.
                for event_id in previews:
                    print(f"span.model_request_end  closing preview for {event_id}")
                previews.clear()
            case "session.status_idle":
                break
'''

In [ ]:
environment = client.beta.environments.create(
    name="python-dev",
    config={
        "type": "cloud",
        "packages": {
            "pip": ["pandas", "numpy", "scikit-learn"],
            "npm": ["express"],
        },
        "networking": {"type": "unrestricted"},
    },
)

print(f"Environment ID: {environment.id}")

In [ ]:
# List environments
environments = client.beta.environments.list()

# Retrieve a specific environment
env = client.beta.environments.retrieve(environment.id)


In [ ]:
session = client.beta.sessions.create(
    agent=agent.id,
    environment_id=environment.id,
    title="Financial analysis on Costco",
    resources=[
        {
            "type": "memory_store",
            "memory_store_id": store.id,
            "access": "read_write",
            "instructions": "User preferences and project context. Check before starting any task.",
        }
    ],
)

print(f"Session ID: {session.id}")


In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
   # Define the outcome — agent starts working on receipt
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.define_outcome",
                "description": "Build a DCF model for Costco in .xlsx",
                #"rubric": {"type": "text", "content": RUBRIC},
                "rubric": {"type": "file", "file_id": rubric.id},
                "max_iterations": 5,  # optional; default 3, max 20
            }
        ],
    )
    # Process streaming events
    for event in stream:
        match event.type:
            case "agent.message":
                for block in event.content:
                    print(block.text, end="")
            case "agent.tool_use":
                print(f"\n[Using tool: {event.name}]")
            case "session.status_idle":
                print("\n\nAgent finished.")
                break

In [ ]:
'''
# Approve the pending tool call
client.beta.sessions.events.send(
    session.id,
    events=[
        {
            "type": "user.tool_confirmation",
            "tool_use_id": event_id,
            "result": "allow",
        },
    ],
    resources=[
        {
            "type": "file",
            "file_id": file.id,
            "mount_path": "/workspace/data.csv",
        },
)


'''

In [ ]:
session = client.beta.sessions.retrieve(session.id)

for outcome in session.outcome_evaluations:
    print(f"{outcome.outcome_id}: {outcome.result}")

In [ ]:
files = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in files.data:
    content = client.beta.files.download(f.id)
    content.write_to_file(f.filename)  # now it's on your machine

In [ ]:
'''
# add file resource to session
resource = client.beta.sessions.resources.add(
    session.id,
    type="file",
    file_id=file.id,
)
print(resource.id) 
'''

In [ ]:
# Archive an environment (read-only, existing sessions continue)
client.beta.environments.archive(environment.id)

# Delete an environment (only if no sessions reference it)
client.beta.environments.delete(environment.id)

In [ ]:
# multi-agent example

# 1. Create the subagents that will be delegated to
reviewer = client.beta.agents.create(
    name="Code Reviewer",
    model="claude-opus-4-8",
    system="You review code for bugs and style issues. Report findings concisely.",
    tools=[{"type": "agent_toolset_20260401"}],
)

test_writer = client.beta.agents.create(
    name="Test Writer",
    model="claude-opus-4-8",
    system="You write unit tests for the code you're given.",
    tools=[{"type": "agent_toolset_20260401"}],
)

# 2. Create the coordinator — `multiagent` is a top-level field, NOT a tools[] entry
coordinator = client.beta.agents.create(
    name="Engineering Lead",
    model="claude-opus-4-8",
    system=(
        "You coordinate engineering work. Delegate code review to the reviewer "
        "and test writing to the test writer. Do the delegation yourself; "
        "don't ask the user for permission first."
    ),
    tools=[{"type": "agent_toolset_20260401"}],
    multiagent={
        "type": "coordinator",
        "agents": [
            reviewer.id,      # bare string = latest version
            test_writer.id,
        ],
    },
)

# 3. Start a session with the coordinator (same as any other session)
environment = client.beta.environments.create(
    name="multiagent-demo",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)

session = client.beta.sessions.create(
    agent=coordinator.id,
    environment_id=environment.id,
    title="Review + test a module",
)
print(f"https://platform.claude.com/workspaces/default/sessions/{session.id}")

# List all threads associated with a session
for thread in client.beta.sessions.threads.list(session_id=session.id):
    print(f"[{thread.agent.name}] {thread.status}")
    with client.beta.sessions.threads.events.stream(thread.id,session_id=session.id) as stream:
        for event in stream:
            match event.type:
                case "agent.message":
                    for block in event.content:
                        if block.type == "text":
                            print(block.text, end="")
                case "session.thread_status_idle":
                    break



In [7]:
deployment = client.beta.deployments.create(
    name="Weekly Email scan",
    agent=agent.id,
    environment_id=environment.id,
    initial_events=[
        {
            "type": "user.message",
            "content": [{"type": "text", "text": "Run the weekly Email scan."}],
        },
    ],
    schedule={
        "type": "cron",
        "expression": "0 20 * * 5",
        "timezone": "America/New_York",
    },
)


NameError: name 'agent' is not defined

In [ ]:
'''
client.beta.deployments.run(deployment.id)

client.beta.deployments.pause(deployment.id)

client.beta.deployments.unpause(deployment.id)

client.beta.deployments.archive(deployment.id)
'''

In [ ]:
uploaded = client.beta.files.upload(
    file=("document.pdf", open("/path/to/document.pdf", "rb"), "application/pdf"),
)
file_id = uploaded.id
print(file_id)